In [ ]:
import numpy as np
import pandas as pd
import shinybroker as sb
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import coint, adfuller
from statsmodels.regression.linear_model import OLS
from statsmodels.tools import add_constant

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.float_format', '{:.4f}'.format)

PAIRS = [
    ('SLB', 'HAL', 'SLB/HAL'),
    ('PSX', 'VLO', 'PSX/VLO'),
    ('XOM', 'CVX', 'XOM/CVX'),
]

FORMATION_START = '2024-03-01'
FORMATION_END   = '2025-03-01'
BACKTEST_START  = '2025-03-01'

ENTRY_Z         =  2.0
EXIT_Z          =  0.0
STOP_Z          =  4.0

TOTAL_CAPITAL   = 100_000
PAIR_CAPITAL    = TOTAL_CAPITAL // len(PAIRS)   # 33333 per pair
COST_BPS        = 5

## Section 2

In [ ]:
import shinybroker as sb

HOST      = '127.0.0.1'
PORT      = 7497        # TWS paper trading
CLIENT_ID = 9999

def make_contract(sym):
    return sb.Contract({
        'symbol': sym,
        'secType': 'STK',
        'exchange': 'SMART',
        'currency': 'USD'
    })

def fetch_daily(sym, duration='2 Y'):
    df = sb.fetch_historical_data(
        contract=make_contract(sym),
        durationStr=duration,
        barSizeSetting='1 day',
        whatToShow='Trades',
        useRTH=True,
        host=HOST,
        port=PORT,
        client_id=CLIENT_ID,
        timeout=60,
    )['hst_dta']
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    return df.set_index('timestamp')[['close']].rename(columns={'close': sym})

tickers = list(set(t for a, b, _ in PAIRS for t in [a, b]))
print(f'Fetching {len(tickers)} tickers...')

raw = {}
for sym in tickers:
    raw[sym] = fetch_daily(sym)
    print(f'  v {sym}')

all_prices = pd.concat(raw.values(), axis=1).dropna()
print(f'\n{len(all_prices)} trading days  ({all_prices.index[0].date()} to {all_prices.index[-1].date()})')
all_prices.head()

## Section 3

In [ ]:
formation = all_prices[FORMATION_START:FORMATION_END].copy()
print(f'Formation: {len(formation)} days  ({formation.index[0].date()} to {formation.index[-1].date()})')
print()

validation = {}

for sym_a, sym_b, name in PAIRS:
    log_a = np.log(formation[sym_a])
    log_b = np.log(formation[sym_b])

    corr = log_a.corr(log_b)
    eg_stat, eg_pval, eg_crits = coint(log_a, log_b)

    ols         = OLS(log_a, add_constant(log_b)).fit()
    alpha       = ols.params.iloc[0]
    hedge_ratio = ols.params.iloc[1]
    spread      = ols.resid

    adf_stat, adf_pval, *_ = adfuller(spread, autolag='AIC')

    validation[name] = {
        'alpha':       alpha,
        'hedge_ratio': hedge_ratio,
        'spread_mean': float(spread.mean()),
        'spread_std':  float(spread.std()),
        'eg_pval':     eg_pval,
        'adf_pval':    adf_pval,
    }

    eg_flag  = 'PASS' if eg_pval  < 0.10 else 'MARGINAL'
    adf_flag = 'PASS' if adf_pval < 0.05 else 'FAIL'

    print(f'{name}')
    print(f'  Correlation  : {corr:.4f}')
    print(f'  EG p-value   : {eg_pval:.4f}  {eg_flag}')
    print(f'  ADF p-value  : {adf_pval:.4f}  {adf_flag}')
    print(f'  Hedge ratio β: {hedge_ratio:.4f}')
    print(f'  R²           : {ols.rsquared:.4f}')
    print()

## Section 4